Few-shot prompting pipeline that uses ONLY the new utils functions:
- load_dataset
- run_prompting (OpenRouter-only, with batching via ThreadPoolExecutor)
- overall_mean, demographic_mean, delta, delta_by_demographic, cohen_kappa, plot_delta_heatmap
- DATASET_CONFIG, DEMOGRAPHICS, OPENROUTER_MODELS

What it does:
1) Loads each dataset
2) Samples N_EXAMPLES rows (fixed seed) to form a few-shot example block
3) Runs each OpenRouter model on the remaining rows with concurrent requests (max_workers)
4) Saves per-model results CSVs
5) Prints metrics + demographic breakdowns
6) Saves delta heatmaps by demographic

- This script runs sequentially across models and datasets, but each model run is parallel across rows
  via run_prompting(max_workers=MAX_WORKERS)

In [ ]:
import os
import pandas as pd

from dotenv import load_dotenv, find_dotenv

from utils import (
    load_dataset,
    run_prompting,
    overall_mean,
    demographic_mean,
    delta,
    delta_by_demographic,
    write_llm_columns_back,
    DATASET_CONFIG,
    DEMOGRAPHICS,
    cohen_kappa,
    OPENROUTER_MODELS,
)

In [2]:
load_dotenv(find_dotenv())

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not set in .env")

print("OPENROUTER_API_KEY loaded:", OPENROUTER_API_KEY[:6] + "...")

OPENROUTER_API_KEY loaded: sk-or-...


In [ ]:
DATA_PATHS = {
    "politeness": "data/raw_data_llm_politeness",
    "offensiveness": "data/raw_data_llm_offensiveness",
}

MAX_ROWS = 1000      # set to None for full dataset
MAX_WORKERS = 50    # number of concurrent API calls per model run

N_EXAMPLES = 10
RANDOM_SEED = 42     # fixed seed so every model sees the same examples

In [4]:
def sample_examples(
    df: pd.DataFrame,
    n: int,
    seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Sample n rows from df to use as few-shot examples, and return:
      (examples_df, remaining_df)
    """
    if n <= 0:
        return df.iloc[0:0].copy(), df.reset_index(drop=True).copy()
    if len(df) <= n:
        # If dataset is tiny, use all as examples and leave nothing to test.
        return df.copy(), df.iloc[0:0].copy()

    examples = df.sample(n=n, random_state=seed)
    remaining = df.drop(examples.index).reset_index(drop=True)
    return examples.reset_index(drop=True), remaining

In [5]:
def format_examples(examples: pd.DataFrame, dataset_name: str) -> str:
    """
    Create the few-shot example block. Uses the human rating column from DATASET_CONFIG.
    """
    config = DATASET_CONFIG[dataset_name]
    text_col = config["text_col"]
    rating_col = config["rating_col"]

    if dataset_name == "politeness":
        header = "Here are example politeness ratings (1=very impolite, 5=very polite):\n"
    else:
        header = "Here are example offensiveness ratings (1=not offensive, 5=extremely offensive):\n"

    lines = [header]
    for _, row in examples.iterrows():
        lines.append(f"Text: {row[text_col]}")
        lines.append(f"Rating: {row[rating_col]}\n")
    return "\n".join(lines)

In [6]:
def make_prompt_builder(dataset_name: str, example_block: str):
    """
    Returns a build_prompt(row)->str function that includes the few-shot examples + the query.
    """
    text_col = DATASET_CONFIG[dataset_name]["text_col"]

    if dataset_name == "politeness":
        question = (
            "Consider you read this email from a colleague. How polite is it?\n"
            "Answer using a 1-5 Likert scale where 1 means 'not polite at all' and 5 means 'very polite'.\n"
            "Return ONLY one integer.\n\n"
        )
    else:
        question = (
            "Consider you read this comment on Reddit. How offensive is it?\n"
            "Answer using a 1-5 Likert scale where 1 means 'not offensive at all' and 5 means 'very offensive'.\n"
            "Return ONLY one integer.\n\n"
        )

    def build_prompt(row: pd.Series) -> str:
        return (
            f"{example_block}\n"
            "Now rate the following:\n"
            f"Text: {row[text_col]}\n\n"
            f"{question}"
        )

    return build_prompt


In [7]:
def smoke_test_models():
    """
    One call to each OpenRouter model to ensure your key + routing works.
    """
    print("\n" + "=" * 80)
    print("SMOKE TEST: 1 call to each OpenRouter model (few-shot style prompt)")
    print("=" * 80)

    df_test = pd.DataFrame({"text": ["Thanks for your help!"], "instance_id": [0], "user_id": [0]})

    # Minimal example block (handmade so it doesn't depend on dataset file existence)
    example_block = (
        "Here are example politeness ratings (1=very impolite, 5=very polite):\n\n"
        "Text: Thanks for sending that over.\n"
        "Rating: 5\n\n"
        "Text: This is unacceptable. Fix it now.\n"
        "Rating: 1\n"
    )
    build_prompt = make_prompt_builder("politeness", example_block)

    failures = []
    for model_label, model_id in OPENROUTER_MODELS.items():
        print(f"\n[SMOKE] {model_label} -> {model_id}")
        try:
            out = run_prompting(
                df=df_test,
                dataset_name="politeness",
                build_prompt_fn=build_prompt,
                model_id=model_id,
                system=None,
                openrouter_api_key=OPENROUTER_API_KEY,
                max_rows=1,
                max_workers=1,
            )
            cols = [c for c in ["llm_text", "llm_rating"] if c in out.columns]
            print(out[cols].to_string(index=False))
        except Exception as e:
            failures.append((model_label, model_id, str(e)))
            print(f"  FAILED: {e}")

    if failures:
        msg = "\n".join([f"- {lbl} ({mid}): {err}" for lbl, mid, err in failures])
        raise RuntimeError("Smoke test failed for some models:\n" + msg)

    print("\nSmoke test passed.\n")
    
smoke_test_models()


SMOKE TEST: 1 call to each OpenRouter model (few-shot style prompt)

[SMOKE] gpt-5.2 -> openai/gpt-5.2
  [gpt-5.2] submitting 1 calls (workers=1)
  [gpt-5.2] 1/1 done
  [gpt-5.2] done — 1 successful, 0 failed
llm_text  llm_rating
       5         5.0

[SMOKE] claude-sonnet-4.6 -> anthropic/claude-sonnet-4.6
  [claude-sonnet-4.6] submitting 1 calls (workers=1)
  [claude-sonnet-4.6] 1/1 done
  [claude-sonnet-4.6] done — 1 successful, 0 failed
llm_text  llm_rating
       5         5.0

[SMOKE] claude-opus-4.6 -> anthropic/claude-opus-4.6
  [claude-opus-4.6] submitting 1 calls (workers=1)
  [claude-opus-4.6] 1/1 done
  [claude-opus-4.6] done — 1 successful, 0 failed
llm_text  llm_rating
       5         5.0

[SMOKE] gemini-3.1-pro -> google/gemini-3.1-pro-preview
  [gemini-3.1-pro-preview] submitting 1 calls (workers=1)
  [gemini-3.1-pro-preview] 1/1 done
  [gemini-3.1-pro-preview] done — 1 successful, 0 failed
llm_text  llm_rating
       5         5.0

[SMOKE] claude-haiku-4.5 -> anthrop

In [8]:
def main():

    all_models = OPENROUTER_MODELS

    # 1) Sample examples + build prompt builders (once per dataset, fixed seed)
    prompt_builders = {}
    remaining_sets = {}

    for dataset_name, data_path in DATA_PATHS.items():
        df_full = load_dataset(data_path)
        config = DATASET_CONFIG[dataset_name]
        text_col = config["text_col"]
        rating_col = config["rating_col"]

        for col in [text_col, rating_col]:
            if col not in df_full.columns:
                raise ValueError(f"{dataset_name}: missing column '{col}' in {data_path}")

        examples_df, remaining_df = sample_examples(df_full, n=N_EXAMPLES, seed=RANDOM_SEED)
        example_block = format_examples(examples_df, dataset_name)
        prompt_builders[dataset_name] = make_prompt_builder(dataset_name, example_block)
        remaining_sets[dataset_name] = remaining_df

        print(f"{dataset_name}: sampled {len(examples_df)} examples, remaining {len(remaining_df)} rows")

    # 2) Run all models — keep results in memory only
    all_results = {ds: {} for ds in DATA_PATHS}

    for dataset_name in DATA_PATHS:
        df = remaining_sets[dataset_name]
        if MAX_ROWS is not None:
            df = df.head(int(MAX_ROWS)).copy()
        build_prompt = prompt_builders[dataset_name]

        for model_label, model_id in all_models.items():
            print(f"\nRunning few-shot: {dataset_name} | {model_label} ({model_id})")
            results_df = run_prompting(
                df=df,
                dataset_name=dataset_name,
                build_prompt_fn=build_prompt,
                model_id=model_id,
                system=None,
                openrouter_api_key=OPENROUTER_API_KEY,
                max_rows=None,
                max_workers=MAX_WORKERS,
            )
            all_results[dataset_name][model_label] = results_df

    # 3) Write model columns back to raw_data_llm.csv (merged by id, no intermediate files)
    for dataset_name, data_path in DATA_PATHS.items():
        print(f"\nWriting columns → {data_path}")
        write_llm_columns_back(
            dataset_name=dataset_name,
            duplicate_input_path=data_path,
            duplicate_output_path=data_path,
            all_model_labels=list(all_models.keys()),
            results_dict=all_results[dataset_name],
            column_suffix="few shot",
        )

    # 4) Print metrics — read from raw_data_llm.csv using the model columns
    for dataset_name, data_path in DATA_PATHS.items():
        human_col = DATASET_CONFIG[dataset_name]["rating_col"]
        merged_df = pd.read_csv(data_path)

        print(f"\n{'='*60}")
        print(f"DATASET: {dataset_name.upper()} (FEW-SHOT)")
        print(f"{'='*60}")

        for model_label in all_models:
            col_name = f"{model_label} (few shot)"
            print(f"\n── {model_label} ──")

            if col_name not in merged_df.columns:
                print(f"  Column {col_name!r} not found, skipping.")
                continue

            print("Overall mean:  ", overall_mean(merged_df, rating_col=col_name))
            print("Delta vs human:", delta(merged_df, human_col, llm_col=col_name))
            print("Cohen's kappa: ", cohen_kappa(merged_df, human_col, llm_col=col_name))

            for demo in DEMOGRAPHICS:
                if demo not in merged_df.columns:
                    print(f"\n  [WARN] {demo} not in columns, skipping.")
                    continue
                print(f"\n  Mean by {demo}:")
                print(demographic_mean(merged_df, demo, rating_col=col_name).to_string(index=False))
                print(f"  Delta by {demo}:")
                print(delta_by_demographic(merged_df, demo, human_col, llm_col=col_name).to_string(index=False))

    print("\nAll done.")

In [9]:
main()

offensiveness: sampled 10 examples, remaining 13026 rows

Running few-shot: offensiveness | gpt-5.2 (openai/gpt-5.2)
  [gpt-5.2] submitting 1000 calls (workers=50)
  [gpt-5.2] 1/1000 done
  [gpt-5.2] 100/1000 done
  [gpt-5.2] 200/1000 done
  [gpt-5.2] 300/1000 done
  [gpt-5.2] 400/1000 done
  [gpt-5.2] 500/1000 done
  [gpt-5.2] 600/1000 done
  [gpt-5.2] 700/1000 done
  [gpt-5.2] 800/1000 done
  [gpt-5.2] 900/1000 done
  [gpt-5.2] 1000/1000 done
  [gpt-5.2] done — 1000 successful, 0 failed

Running few-shot: offensiveness | claude-sonnet-4.6 (anthropic/claude-sonnet-4.6)
  [claude-sonnet-4.6] submitting 1000 calls (workers=50)
  [claude-sonnet-4.6] 1/1000 done
  [claude-sonnet-4.6] 100/1000 done
  [claude-sonnet-4.6] 200/1000 done
  [claude-sonnet-4.6] 300/1000 done
  [claude-sonnet-4.6] 400/1000 done
  [claude-sonnet-4.6] 500/1000 done
  [claude-sonnet-4.6] 600/1000 done
  [claude-sonnet-4.6] 700/1000 done
  [claude-sonnet-4.6] 800/1000 done
  [claude-sonnet-4.6] 900/1000 done
  [claud

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import sem, t as t_dist

METRICS_DATA_PATHS = {
    "politeness":    "../../dataset/politeness_rating/raw_data_llm.csv",
    "offensiveness": "../../dataset/offensiveness/raw_data_llm.csv",
}
HUMAN_COLS = {"politeness": "politeness", "offensiveness": "offensiveness"}
PROMPT_TYPE = "few shot"
N_ROWS = 1000

MODELS = [
    "gpt-5.2", "claude-sonnet-4.6", "claude-opus-4.6", "gemini-3.1-pro",
    "claude-haiku-4.5", "llama-3-8b", "mistral-large-2512", "gpt-oss-120b",
]

GENDER_GROUPS    = ["Man", "Woman", "Non-binary"]
EDUCATION_GROUPS = [
    "Less than a high school diploma", "High school diploma or equivalent",
    "College degree", "Graduate degree",
]
AGE_GROUPS   = ["18-29", "30-39", "40-49", "50-59", ">=60"]
DEMOGRAPHICS = [
    ("gender",    "gender",    GENDER_GROUPS),
    ("education", "education", EDUCATION_GROUPS),
    ("age",       "age_group", AGE_GROUPS),
]

def _bin_age(val):
    s = str(val).strip().lstrip(">").split("-")[0].replace("+", "").strip()
    try:
        age = float(s)
    except ValueError:
        return "Unknown"
    if age < 18:  return "Unknown"
    if age <= 29: return "18-29"
    if age <= 39: return "30-39"
    if age <= 49: return "40-49"
    if age <= 59: return "50-59"
    return ">=60"

def _ci95(series):
    series = pd.to_numeric(series, errors="coerce").dropna()
    n = len(series)
    if n < 2:
        return np.nan, np.nan
    margin = t_dist.ppf(0.975, df=n - 1) * sem(series)
    return series.mean() - margin, series.mean() + margin

def _fmt(series):
    series = pd.to_numeric(series, errors="coerce").dropna()
    if len(series) == 0:
        return "N/A"
    mu = series.mean()
    lo, hi = _ci95(series)
    return f"{mu:.4f} ({lo:.4f}, {hi:.4f})"

def _mean(series):
    series = pd.to_numeric(series, errors="coerce").dropna()
    return series.mean() if len(series) > 0 else np.nan

for dataset_name, data_path in METRICS_DATA_PATHS.items():
    df = pd.read_csv(data_path).head(N_ROWS).copy()
    human_col = HUMAN_COLS[dataset_name]
    if "age" in df.columns:
        df["age_group"] = df["age"].apply(_bin_age)

    print(f"\n{'='*80}")
    print(f"  DATASET: {dataset_name.upper()} — FEW-SHOT METRICS (first {N_ROWS} rows)")
    print(f"{'='*80}")

    # 1. Ground truth mean + CI95 per demographic
    print("\n── 1. GROUND TRUTH: Mean (CI95) per demographic group ──")
    for src_col, display_col, groups in DEMOGRAPHICS:
        if display_col not in df.columns:
            print(f"\n  [SKIP] '{display_col}' not in dataset")
            continue
        label = "AGE GROUP" if src_col == "age" else src_col.upper()
        rows = []
        for g in groups:
            subset = df[df[display_col] == g][human_col]
            n = int(pd.to_numeric(subset, errors="coerce").count())
            if n == 0:
                continue
            rows.append({"Group": g, "N": n, "Mean (CI95)": _fmt(subset)})
        print(f"\n  {label}")
        print(pd.DataFrame(rows).to_string(index=False))

    # 2. Model mean + CI95 overall
    print("\n\n── 2. MODEL MEAN (CI95) OVERALL ──")
    model_rows = []
    model_overall_mean = {}
    for model_label in MODELS:
        col_name = f"{model_label} ({PROMPT_TYPE})"
        if col_name not in df.columns:
            model_rows.append({"Model": model_label, "N": 0, "Mean (CI95)": "MISSING"})
            model_overall_mean[model_label] = np.nan
            continue
        series = df[col_name]
        model_overall_mean[model_label] = _mean(series)
        model_rows.append({
            "Model": model_label,
            "N": int(pd.to_numeric(series, errors="coerce").count()),
            "Mean (CI95)": _fmt(series),
        })
    print(pd.DataFrame(model_rows).to_string(index=False))

    # 3. Delta = GT group mean − model overall mean
    print("\n\n── 3. DELTA: GT group mean − model overall mean ──")
    print("   (negative = model rates higher; positive = model rates lower)")
    for src_col, display_col, groups in DEMOGRAPHICS:
        if display_col not in df.columns:
            continue
        label = "AGE GROUP" if src_col == "age" else src_col.upper()
        print(f"\n  {label}")
        delta_rows = []
        abs_delta_by_model = {m: [] for m in MODELS}
        for g in groups:
            subset = df[df[display_col] == g]
            if len(subset) == 0:
                continue
            gt_mean = _mean(subset[human_col])
            row = {"Group": g, "GT Mean": f"{gt_mean:.4f}" if pd.notna(gt_mean) else "N/A"}
            for model_label in MODELS:
                m_mean = model_overall_mean[model_label]
                if pd.notna(gt_mean) and pd.notna(m_mean):
                    d = gt_mean - m_mean
                    row[model_label] = f"{d:+.4f}"
                    abs_delta_by_model[model_label].append(abs(d))
                else:
                    row[model_label] = "N/A"
            delta_rows.append(row)
        print(pd.DataFrame(delta_rows).to_string(index=False))
        mean_abs = {m: np.mean(v) if v else np.nan for m, v in abs_delta_by_model.items()}
        valid = {m: v for m, v in mean_abs.items() if pd.notna(v)}
        if valid:
            best = min(valid, key=valid.get)
            ranking = sorted(valid.items(), key=lambda x: x[1])
            print(f"  → Least mean |Δ|: {best}  ({valid[best]:.4f})")
            print(f"     Ranking: {[(m, round(v,4)) for m,v in ranking]}")


  DATASET: POLITENESS — FEW-SHOT METRICS (first 1000 rows)

── 1. GROUND TRUTH: Mean (CI95) per demographic group ──

  GENDER
     Group   N             Mean (CI95)
       Man 549 3.2842 (3.1873, 3.3810)
     Woman 401 3.1122 (2.9661, 3.2583)
Non-binary  50 3.1000 (2.7842, 3.4158)

  EDUCATION
                            Group   N             Mean (CI95)
High school diploma or equivalent 300 3.1867 (3.0404, 3.3329)
                   College degree 501 3.0539 (2.9372, 3.1706)
                  Graduate degree 149 3.7517 (3.5544, 3.9490)

  AGE GROUP
Group   N             Mean (CI95)
18-29 251 3.0000 (2.8263, 3.1737)
30-39 349 3.0516 (2.9116, 3.1915)
40-49 350 3.4543 (3.3276, 3.5810)
50-59  50 3.5800 (3.3645, 3.7955)


── 2. MODEL MEAN (CI95) OVERALL ──
             Model    N             Mean (CI95)
           gpt-5.2 1000 3.1410 (3.0735, 3.2085)
 claude-sonnet-4.6 1000 3.1240 (3.0552, 3.1928)
   claude-opus-4.6 1000 3.1520 (3.0831, 3.2209)
    gemini-3.1-pro  974 2.9487 (2.8674, 3.0